# Análisis de Riesgo y Rentabilidad

Este archivo realiza un análisis del riesgo y la rentabilidad utilizando datos de acciones. Incluye pasos como carga de datos, limpieza, análisis estadístico y visualización de resultados.

## Importación de Librerías

Se importan las librerías necesarias para el análisis: `pandas` y `numpy`.

In [ ]:
import pandas as pd
import numpy as np

## Carga de Datos

Cargamos los datos desde archivos Excel utilizando pandas. Los datos se combinan en un único DataFrame para facilitar el análisis.

In [ ]:
df = pd.read_excel('../data/processed/Datos_Acciones.xlsx', header=0, index_col=None)

# Limpieza de Datos

Creamos un df de rentabilidad y eliminamos las columnas que no nos interesan

In [ ]:
df_rent = df.copy()
df_rent.drop(df.columns[3], axis = 1, inplace= True)


Ahora tenemos en nuestro df los valores de las acciones cada mes. A través de ello tenemos que obtener la rentabilidad mensual

#### Función: `rent_m`

Esta función calcula la rentabilidad mensual de un fondo, a partir de la diferencia porcentual entre el valor de la acción de un mes y el mes anterior. La rentabilidad se calcula como `(valor mes actual - valor mes anterior) / valor mes anterior`, y se aplica de forma iterativa a lo largo de las filas de un DataFrame.


In [ ]:
def rent_m(row: pd.Series):
    resultado = row.copy()    
    for i in range(3, len(row)-1):
        resultado.iloc[i] = ((row.iloc[i+1]-row.iloc[i])/row.iloc[i])
    return resultado

df_rent= df_rent.apply(lambda x: rent_m(x), axis = 1)

Como la rentabilidad hace referencia al mes y no al primer dia del mes, cambiamos el nombre de las columnas

In [ ]:
df_rent.columns = df_rent.columns.str.replace('^1 ', '', regex=True)

Eliminamos la columna de Diciembre de 2024, ya que como no tenemos los datos de enero, no podemos saber la rentabilidad de diciembre


In [ ]:
df_rent = df_rent.drop('Dec 2024', axis = 1)

Convertimos la segunda y tercera columna en números para poder operar con ellos. En vez de porcentajes, como rendimientos

In [ ]:
df_rent.iloc[:, 1:3] = df_rent.iloc[:, 1:3].apply(lambda x: x.str.replace('.', '', regex=False)
                                                            .str.replace(',', '.', regex=False)
                                                            .str.replace('%', '', regex=False) 
                                                            .astype(float)
                                                            /100
                                                )

Eliminamos esta columna ya que tiene datos erróneos y la calcularemos de nuevo a continuación

In [ ]:
df_rent.drop('Rentabilidad 5 años', inplace = True, axis= 1)

#### Función: `calc_rent_vol`

Esta función calcula la rentabilidad acumulada y la volatilidad de un fondo en función de los años indicados. Se puede especificar si se desea calcular la rentabilidad acumulada o la volatilidad del fondo. La rentabilidad se calcula como el producto de las rentabilidades mensuales, mientras que la volatilidad se calcula como la desviación estándar de las rentabilidades mensuales, ajustada por la raíz cuadrada de 12 para anualizarla.

##### Parámetros:
- `row`: Serie de Pandas que contiene las rentabilidades mensuales.
- `años`: Número de años para calcular la rentabilidad acumulada o volatilidad.
- `rent`: Si es 1, calcula la rentabilidad acumulada.
- `vol`: Si es 1, calcula la volatilidad.

##### Retorna:
- Si `rent == 1`: Retorna la rentabilidad acumulada durante los años indicados.
- Si `vol == 1`: Retorna la volatilidad anualizada durante los años indicados.
- Si ninguno de los dos está activado o si las condiciones no se cumplen, retorna `np.nan`.


In [ ]:
def calc_rent_vol(row: pd.Series, años: int, rent: int = 0, vol: int = 0):
    '''
    Dada una fila con sus rentabilidades mensuales en función de lo que se indique calcula:
        - La rentabilidad acumulada de la cantidad de años indicada
        - La volatilidad en el periodo de años indicado
    '''
    if años < 6: 
        valor = row[f'Jan {2025 - años}']
        if pd.isnull(valor) or valor == 0:
            return np.nan
        elif rent == 1:
            return np.prod(row + 1) - 1
        elif vol == 1:
            return row.std()*np.sqrt(12)
        else:
            return np.nan


Añadimos al dataframe las columnas de la rentabilidad acumulada por periodos de 1, 3 y 5 años

In [ ]:
value_1_año = df_rent.loc[:, 'Jan 2024':].apply(calc_rent_vol, años = 1, rent= 1, axis = 1)
value_3_años = df_rent.loc[:, 'Jan 2022':].apply(calc_rent_vol, años = 1, rent= 1, axis = 1)
value_5_años = df_rent.loc[:, 'Jan 2020':].apply(calc_rent_vol, años = 1, rent= 1, axis = 1)
df_rent.insert(1, 'Rentabilidad 1 año', value_1_año)
df_rent.insert(2, 'Rentabilidad 3 años', value_3_años)
df_rent.insert(3, 'Rentabilidad 5 años', value_5_años)

Eliminamos las filas que no tienen datos para ninguno de los 3 periodos

In [ ]:
df_rent.dropna(subset=['Rentabilidad 1 año', 'Rentabilidad 3 años', 'Rentabilidad 5 años'], how= 'all', inplace= True)

Añadimos al dataframe las columnas de la volatilidad anualizada por periodos de 1, 3 y 5 años

In [ ]:
v_1_año = df_rent.loc[:, 'Nov 2023':].apply(calc_rent_vol,años = 1, vol= 1, axis = 1)
v_3_años = df_rent.loc[:, 'Nov 2021':].apply(calc_rent_vol,años = 3, vol= 1, axis = 1)
v_5_años = df_rent.loc[:, 'Jan 2020':].apply(calc_rent_vol,años = 5, vol= 1, axis = 1)
df_vol = df_rent.copy()
df_vol['1 año'] = v_1_año
df_vol['3 años'] = v_3_años
df_vol['5 años'] = v_5_años

Creamos un dataframe con indice de columnas multinivel con los datos de rentabilidad mensual, y volatilidad por periodos

In [ ]:
level1_cols = ['ACCIÓN'] + ['RENTABILIDAD']*(len(df_rent.columns) - 1) + ['VOLATILIDAD']*3
df_RyV_month = pd.DataFrame(df_vol.values, columns=[level1_cols, df_vol.columns])

Separamos los datos de volatilidad en 4 rangos, para que sea más visual según el nivel de riesgo

In [ ]:
años = ['1 año', '3 años','5 años']

for i in range(3):
    Q1 = df_RyV_month[('VOLATILIDAD', años[i])].quantile(0.25)
    Q2 = df_RyV_month[('VOLATILIDAD', años[i])].quantile(0.5)
    Q3 = df_RyV_month[('VOLATILIDAD', años[i])].quantile(0.75)
    df_RyV_month[('RIESGO', años[i])] = df_RyV_month[('VOLATILIDAD', años[i])].map(lambda x: "MUY ALTO" if x > Q3 else "ALTO" if Q2 < x <= Q3 else "MEDIO" if Q1 < x <= Q2 else "BAJO" if x < Q2 else np.nan)

Reseteamos los indices para que a parezcan en orden correcto (por si ha variado al eliminar filas)

In [ ]:
df_RyV_month.reset_index(drop=True, inplace= True)

Exportamos el excel limpio, con datos de rentabilidad acumulada y volatilidad anuealizada en periodos de 1, 3 y 5 años. 

Tambien contiene el nivel de riesgo: bajo, medio, alto o muy alto

In [ ]:
df_RyV_month.to_excel("../data/processed/Volatilidad_rentabilidad.xlsx")

# Preparación de los Datos

Se seleccionan las columnas relevantes para el análisis.

In [ ]:
cols = [('RENTABILIDAD', 'Rentabilidad 1 año'), ('RENTABILIDAD', 'Rentabilidad 3 años'), ('RENTABILIDAD', 'Rentabilidad 5 años'),
        ('VOLATILIDAD', '1 año'), ('VOLATILIDAD', '3 años'), ('VOLATILIDAD', '5 años')]

Se convierten los valores a porcentaje para la representación sea más visual.

In [ ]:
df_RyV_month[cols] = df_RyV_month[cols].apply(lambda x: x*100)

## Importación de Librerías

Se importan las librerías necesarias para la visualización: `matplotlib.pyplot`.

In [ ]:
import matplotlib.pyplot as plt

# Visualización de los datos obtenidos

### Gráfico: Rentabilidad vs. Volatilidad, según los niveles de riesgo

Este gráfico de dispersión compara la rentabilidad frente a la volatilidad de las acciones, categorizándolas según su nivel de riesgo: Bajo, Medio, Alto y Muy Alto. 

Cada grupo de fondos se representa con un color diferente, lo que facilita la visualización de cómo se distribuyen los fondos según el riesgo en función de su rentabilidad y volatilidad.

In [ ]:
color_map = {'BAJO':'green',
             'MEDIO':'blue',
             'ALTO':'darkorange',
             'MUY ALTO': 'red'}

ax = plt.gca()

grouped = df_RyV_month.groupby(('RIESGO', '5 años'))
for key, group in grouped:
    group.plot(kind= 'scatter', 
                x = ('VOLATILIDAD', '5 años'), 
                y = ('RENTABILIDAD', 'Rentabilidad 5 años'),
                xlabel='Volatilidad (%)', 
                ylabel= 'Rentabilidad (%)',
                label=key, 
                title= 'Volatilidad vs Rentabilidad',
                ax = ax,
                color = color_map[key],
                figsize=(12, 8))
plt.legend(title="Nivel de Riesgo")
plt.grid(True)

Vemos que hay un outlier, lo eliminamos para ver mejor la gráfica

In [ ]:
df_sin_outliers = df_RyV_month.sort_values(by = [('RENTABILIDAD', 'Rentabilidad 5 años')], ascending= False)
df_sin_outliers.reset_index(drop=True, inplace= True)
df_sin_outliers.drop(index = [0, 1], inplace= True)


Repetimos la gráfica sin el outlier

In [ ]:
color_map = {'BAJO':'green',
             'MEDIO':'blue',
             'ALTO':'darkorange',
             'MUY ALTO': 'red'}

ax = plt.gca()

grouped = df_sin_outliers.groupby(('RIESGO', '5 años'))
for key, group in grouped:
    group.plot(kind= 'scatter', 
                x = ('VOLATILIDAD', '5 años'), 
                y = ('RENTABILIDAD', 'Rentabilidad 5 años'),
                xlabel='Volatilidad (%)', 
                ylabel= 'Rentabilidad (%)', 
                title= 'Volatilidad vs Rentabilidad',
                label=key,
                ax = ax,
                color = color_map[key],
                figsize=(12, 8))
    plt.legend(title="Nivel de Riesgo")
    plt.grid(True)

plt.tight_layout()
plt.savefig("../reports/figures/stocks/stocks_risk_return_scatter.png", dpi=300, bbox_inches="tight")

### Gráfico: Rentabilidad por Nivel de Riesgo

Este gráfico de cajas (boxplot) muestra la distribución de la rentabilidad acumulada de los últimos 5 años de las acciones, según su nivel de riesgo. 

Cada grupo de riesgo está representado por una caja que permite visualizar la mediana, el rango intercuartílico, y los valores atípicos. 

In [ ]:
color_map = {'BAJO':'green',
             'MEDIO':'blue',
             'ALTO':'darkorange',
             'MUY ALTO': 'red'}
fig, ax = plt.subplots(figsize=(12, 8))
grouped = df_RyV_month.groupby(('RIESGO', '5 años'))
orden = ['BAJO', 'MEDIO', 'ALTO', 'MUY ALTO']
count = 0
for key in orden:
    grouped.get_group(orden[count])[('RENTABILIDAD', 'Rentabilidad 5 años')].plot( 
        kind= 'box',
        label = key,
        positions=[count],
        figsize=(12, 8))
    count += 1

ax.set_xticks(range(len(orden)))
ax.set_xticklabels(orden)
ax.set_xlabel('RIESGO')
ax.set_ylabel('RENTABILIDAD (%)')

Vemos que hay varios outliers, los eliminamos para ver mejor la gráfica. La repetimos una vez eliminados

In [ ]:
# Elimino los outliers
fig, ax = plt.subplots(figsize=(12, 8))
count = 0
for key in orden:
    grouped.get_group(orden[count])[('RENTABILIDAD', 'Rentabilidad 5 años')].plot( 
        kind= 'box',
        label = key,
        positions=[count],
        showfliers=False,
        figsize=(12, 8),
        )
    count += 1

ax.set_xticks(range(len(orden)))
ax.set_xticklabels(orden)
ax.set_xlabel('RIESGO')
ax.set_ylabel('RENTABILIDAD (%)')

plt.tight_layout()
plt.savefig("../reports/figures/stocks/stocks_return_by_risk_boxplot.png", dpi=300, bbox_inches="tight")